# RQ2 immutable-window rerun

This notebook reruns all 100 Revision 5 questions and all 36 Revision 4 questions for both answer paths. Both paths receive the same ordered, text-hashed evidence window. Gold labels are unavailable to the model and are used only by the verifier.

Before selecting **Runtime → Run all**:

1. Select a T4 GPU or better under **Runtime → Change runtime type**.
2. Create a fine-grained GitHub token whose resource owner is `Wenjia1215`, whose selected repository is `ComplianceGPT_v2`, and whose repository permission is **Contents: Read-only**. Add it as a Colab secret named `GITHUB_TOKEN` and enable notebook access. Never paste the token into notebook code. If the secret is absent, the notebook asks for it without displaying it.
3. Approve the Google Drive mount. Results are checkpointed after every question, so the same notebook and output folder can resume after a disconnect.
4. The notebook owns its disposable clone under `/content` and refreshes it automatically when this branch advances. Drive checkpoints are preserved.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU not found. Choose Runtime > Change runtime type > T4 GPU, then rerun.'
)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import base64
import getpass
import shutil
import subprocess
import urllib.error
import urllib.request
from pathlib import Path

from google.colab import userdata

REPOSITORY = 'Wenjia1215/ComplianceGPT_v2'
BRANCH = 'codex/rq2-immutable-window-v2'
REPO_DIR = Path('/content/ComplianceGPT_v2_rq2')

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None
if not github_token:
    github_token = getpass.getpass('GitHub token (read-only repository access is sufficient): ')
if not github_token:
    raise RuntimeError('A GitHub token is required to clone the private repository.')

api_url = f'https://api.github.com/repos/{REPOSITORY}/contents/README.md?ref={BRANCH}'
api_request = urllib.request.Request(
    api_url,
    headers={
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {github_token}',
        'X-GitHub-Api-Version': '2022-11-28',
    },
)
permission_error = None
try:
    with urllib.request.urlopen(api_request, timeout=30) as response:
        if response.status != 200:
            permission_error = f'HTTP {response.status}'
except urllib.error.HTTPError as error:
    permission_error = f'HTTP {error.code}'
except urllib.error.URLError as error:
    permission_error = f'network error: {error.reason}'
api_request = None
if permission_error:
    github_token = None
    raise RuntimeError(
        'The token cannot read the selected private repository (' + permission_error + '). '
        'Create a new fine-grained token with resource owner Wenjia1215, select '
        'ComplianceGPT_v2, and grant Repository permissions > Contents: Read-only.'
    )

basic = base64.b64encode(f'Wenjia1215:{github_token}'.encode()).decode()
auth_option = f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}'
remote_url = f'https://github.com/{REPOSITORY}.git'

remote_result = subprocess.run(
    ['git', '-c', auth_option, 'ls-remote', remote_url, f'refs/heads/{BRANCH}'],
    text=True, capture_output=True,
)
if remote_result.returncode != 0:
    remote_error = remote_result.stderr
    remote_result = None
    github_token = None
    basic = None
    auth_option = None
    raise RuntimeError('Could not resolve the runner branch: ' + remote_error)
remote_line = remote_result.stdout.strip()
remote_result = None
remote_head = remote_line.split()[0] if remote_line else ''
if not remote_head:
    github_token = None
    basic = None
    auth_option = None
    raise RuntimeError(f'Branch not found: {BRANCH}')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        shutil.rmtree(REPO_DIR)
    else:
        local_head = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
        ).strip()
        if local_head != remote_head:
            print('Refreshing the disposable clone to:', remote_head)
            shutil.rmtree(REPO_DIR)

if not (REPO_DIR / '.git').is_dir():
    result = subprocess.run(
        ['git', '-c', auth_option, 'clone', '--branch', BRANCH, '--single-branch', remote_url, str(REPO_DIR)],
        text=True, capture_output=True
    )
    if result.returncode != 0:
        clone_error = result.stderr
        result = None
        github_token = None
        basic = None
        auth_option = None
        if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
            shutil.rmtree(REPO_DIR)
        raise RuntimeError('Repository clone failed after the permission check: ' + clone_error)
    result = None

github_token = None
basic = None
auth_option = None
print('Runner commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())


In [ ]:
import subprocess
import sys

requirements = REPO_DIR / 'experiments/answerer_comparison/rq2_matched_v2/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Dependencies installed.')


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/compliancegpt_rq2_matched_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint directory:', OUTPUT_DIR)


In [ ]:
import subprocess
import sys
from collections import deque

runner = REPO_DIR / 'experiments/answerer_comparison/rq2_matched_v2/run_matched_rq2.py'
command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
tail = deque(maxlen=200)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    tail.append(line)
return_code = process.wait()
if return_code != 0:
    failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
    failure_log.write_text(''.join(tail), encoding='utf-8')
    print(f'\nRQ2 runner failed. Its final output is saved at {failure_log}.')
    raise subprocess.CalledProcessError(return_code, command)
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
if failure_log.exists():
    failure_log.unlink()
print('RQ2 runner completed successfully.')


In [ ]:
import csv
import json
import shutil
from google.colab import files

expected_rows = {
    'rev5_compliancegpt.csv': 100,
    'rev4_compliancegpt.csv': 36,
    'rev5_generative_baseline.csv': 100,
    'rev4_generative_baseline.csv': 36,
}
contracts_dir = OUTPUT_DIR / 'contracts'
problems = []
for filename, expected in expected_rows.items():
    path = contracts_dir / filename
    if not path.is_file():
        problems.append(f'{filename}: missing; expected {expected}')
        continue
    with path.open('r', encoding='utf-8', newline='') as handle:
        rows = sum(1 for _ in csv.DictReader(handle))
    if rows != expected:
        problems.append(f'{filename}: {rows} rows; expected {expected}')
    else:
        print(f'{filename}: {rows} rows')

paired_path = OUTPUT_DIR / 'manifests/paired_validation.json'
if not paired_path.is_file():
    problems.append('Paired-validation manifest is missing.')
else:
    paired = json.loads(paired_path.read_text(encoding='utf-8'))
    for revision, expected in {'rev5': 100, 'rev4': 36}.items():
        validation = paired.get(revision, {}).get('validation', {})
        if validation.get('paired_rows') != expected:
            problems.append(
                f'{revision} paired rows: {validation.get("paired_rows")}; expected {expected}'
            )
        for field in (
            'context_mismatches',
            'evidence_window_mismatches',
            'model_mismatches',
        ):
            if validation.get(field) != 0:
                problems.append(f'{revision} {field}: {validation.get(field)}; expected 0')

if problems:
    print('\n'.join(problems))
    raise RuntimeError(
        'The RQ2 run is incomplete or failed paired validation. '
        'Rerun the runner cell; valid completed questions will be skipped.'
    )

result_zip = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
if not result_zip.is_file():
    result_zip = Path(shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR))
print('Validated result archive:', result_zip)
files.download(str(result_zip))
